Diffusers and pipeline Code

In [1]:
from diffusers import StableDiffusion3Pipeline
import torch
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

    # ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Imports, configs

In [2]:
from pathlib import Path
import gc
import pandas as pd
from tqdm import tqdm
import shutil

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/tsr_samples_tester_new")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/pt_samples_tester_new")

REPLICA_EXCHANGE = True
LAM_VALUES = [1.2]
INDEX_UNTIL = 6

gc.collect()
torch.cuda.empty_cache()

SWAP_ALGORITHM = {
	"n_replicas": 4,
	"p_ratio": "p",
	"low_indices": [22, 23],
	"mid_indices":  [9, 14],
	"high_indices":  [0, 7],
	"debug": True,
}

if PT_TSR_DIR.exists():
    shutil.rmtree(PT_TSR_DIR)
    

Sample!

In [ ]:
# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")


replica_exchanges = [False, True]

lam_dirs = {}
for re in replica_exchanges:
	base = PT_TSR_DIR if re else TSR_DIR
	lam_dirs[re] = {l: base / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
	for d in lam_dirs[re].values():
		d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	
	for replica_exchange in replica_exchanges:
	
		for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx} re={replica_exchange}"):

			output_dir = lam_dirs[replica_exchange][tsr_lam]

			if (output_dir / f"{idx:05d}.png").exists():
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED)

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=TSR_SIGMA,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images

			out_path = output_dir / f"{idx:05d}.png"
			images[0].save(out_path, icc_profile=None)


			del images
		torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 6 prompts


idx=0 re=True:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.20 with replica exchange True
Time 1000.00 swap btwn source 4.29 and target 1.20 accept 1.000 std 0.984
Time 904.45 swap btwn source 4.29 and target 1.20 accept 0.990 std 0.904
Time 870.06 swap btwn source 1.88 and target 1.20 accept 1.000 std 0.885
Time 763.76 swap btwn source 1.88 and target 1.20 accept 0.973 std 0.848
Time 491.46 swap btwn source 0.88 and target 1.20 accept 0.998 std 0.860
Time 442.58 swap btwn source 0.88 and target 1.20 accept 0.996 std 0.877


idx=1 re=True:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.20 with replica exchange True
Time 1000.00 swap btwn source 4.29 and target 1.20 accept 1.000 std 0.985
Time 904.45 swap btwn source 4.29 and target 1.20 accept 0.992 std 0.902
Time 870.06 swap btwn source 1.88 and target 1.20 accept 1.000 std 0.880
Time 763.76 swap btwn source 1.88 and target 1.20 accept 0.976 std 0.834


Fid computation

In [ ]:
from fid import compute_sweep

LAM_VALUES = [1.2, 1.1, 1.05, 1.01]

compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True, False],
	device="cuda",
	target_indices=None,
	index_until=INDEX_UNTIL,
	tsr_dir = TSR_DIR,
	pt_sr_dir = PT_TSR_DIR,
)